# Suite TRF — Transforms

A transform is a rewrite whose `before` and `after` are predicates over the same symbols, with parameters typed by value
domains. A session keeps the candidates: steps enabled where `before` holds and `after` does not, one per value of each
finite domain, ordered with parameter decisions first. It takes one step per decision, as the caller decides or, stepping
over, as a policy ranks; a composite transform is stepped into or over. The steps taken are a trace, written as data.

In [ ]:
import { Expressions as E } from "@mbse/expressions";
import { Predicates as P, Transforms as T } from "@mbse/patterns";
import { JSON as SchemaJSON, Proxies, Reflection as R, Schemas as S, Validators as SV } from "@mbse/schemas/Framework";
import { ValueError } from "@mbse/schemas/Framework/Errors";
import { assert, equal, native, raises, same } from "./support.js";

const Held = new S.OfRelation.Builder().name("Held").links("shelf", "item").create();
const Shelf = new S.OfObject.Builder().name("Shelf").ref().singleton("Shelf").relations((r) => r.name("items").of(Held).me("shelf")).create();
const Item = new S.OfObject.Builder().name("Item").ref().properties(native("name"), native("label"), native("size", BigInt)).relations(
  (r) => r.name("shelves").of(Held).me("item")).create();
const i = E.variable("i");

/** A predicate over one item, `i`. */
function over(constraint: unknown = null): P.OfPredicate {
  const builder = new P.OfPredicate.Builder().symbols({ i: Item });
  return (constraint === null ? builder : builder.requires(constraint)).create();
}

/** A store whose shelf holds an item of each name, in order (null for an item without a name). */
function shelf(...names: (string | null)[]): any {
  const store: any = new Proxies.OfStore();
  for (const schema of [Shelf, Item, Held]) store.register(schema);
  for (const name of names) {
    const item = name === null ? store.Item().create() : store.Item().name(name).create();
    store.Shelf(store.singleton("Shelf")).items((e: any) => e.item(item)).update();
  }
  return store;
}

const Option = new S.OfObject.Builder().create();
const Case = new S.OfUnion.Builder().branches((b) => b.name("upper").of(Option), (b) => b.name("lower").of(Option)).create();

function label(store: any, match: T.Match, args: Record<string, unknown>): void {
  const item = match["i"] as any;
  store.Item(item).label(args["case"] === "upper" ? item.name.toUpperCase() : item.name.toLowerCase()).update();
}

function size(store: any, match: T.Match, args: Record<string, unknown>): void {
  store.Item(match["i"]).size(args["size"]).update();
}

const Label = new T.Transform("Label", over(i.has("name")), over(i.has("label")), { parameters: [(p) => p.name("case").of(Case)], rewrite: label });
const Size = new T.Transform("Size", over(i.has("label")), over(i.has("size")),
  { parameters: [(p) => p.name("size").of((t) => t.as_native(BigInt))], rewrite: size });
const Tag = new T.Transform("Tag", over(i.has("name")), over(i.has("label")),
  { rewrite: (store, match) => store.Item(match["i"]).label("tag").update() });
const Finish = new T.Transform("Finish", over(i.has("name")), over(i.has("label").and_(i.has("size"))), { parts: [Label, Size] });

/** Each candidate as its transform, item and arguments. */
function seen(candidates: T.Candidate[]): unknown[] {
  return candidates.map((c) => [c.transform.name, (c.match["i"] as any).name, c.arguments]);
}

## TRF-01 · A transform: before and after over the same symbols, parameters typed by domains, a rewrite or parts

In [ ]:
{
  assert(Label.name === "Label" && same(Label.parameters.keys(), ["case"]) && Label.parameters.get("case")?.type === Case && Label.parts.length === 0);
  assert(Label.check().length === 0 && Finish.check().length === 0 && Finish.rewrite === null && same(Finish.parts, [Label, Size]));
  assert(same(new T.Transform("X", over(), over(), { parameters: [new S.OfParameter.Data({ name: "n" })], rewrite: size }).parameters.keys(), ["n"])); // data or specs
  const odd = new T.Transform("Odd", over(), new P.OfPredicate.Builder().symbols({ j: Item }).create(), { rewrite: size });
  assert(same(odd.check(), ["transform 'Odd': before and after must have the same symbols"]));
  assert(same(new T.Transform("Bare", over(), over()).check(), ["transform 'Bare': a transform has a rewrite or parts, one of them"]));
  assert(same(new T.Transform("Both", over(), over(), { rewrite: size, parts: [Label] }).check(), [
    "transform 'Both': a transform has a rewrite or parts, one of them"]));
  assert(same(new T.Transform("Outer", over(), over(), { parts: [odd] }).check(), ["transform 'Odd': before and after must have the same symbols"]));
  raises(ValueError, () => new T.Session(shelf(), [odd, new T.Transform("Bare", over(), over())]),
    "transform 'Odd': before and after must have the same symbols; transform 'Bare': a transform");
}

## TRF-02 · Candidates: enabled where before holds and after does not, finite domains expanded, parameter decisions first

In [ ]:
const Option2 = new S.OfUnion.Builder().branches((b) => b.name("a").of(Item), (b) => b.name("b").of(Option)).create();
const Free = new T.Transform("Free", over(i.has("name")), over(i.has("size")), {
  parameters: [(p) => p.name("n").of(Option2), (p) => p.name("on").of((t) => t.as_native(Boolean))], rewrite: size });
const free = new T.Session(shelf("bolt"), [Free]).candidates();
{
  const store = shelf("bolt", null, "nut");
  const Big = new T.Transform("Big", over(i.get("size").gt(2n)), over(i.has("label")), { rewrite: size });
  const session = new T.Session(store, [Tag, Label, Size, Big]);
  assert(equal(seen(session.candidates()), [ // Label's, which decide a parameter, first; an item without a name matches neither
    ["Label", "bolt", { case: "upper" }], ["Label", "bolt", { case: "lower" }],
    ["Label", "nut", { case: "upper" }], ["Label", "nut", { case: "lower" }],
    ["Tag", "bolt", {}], ["Tag", "nut", {}]]));
  assert(same(session.undecided, ["'Big' at i=Item#0: its before is unknown", "'Big' at i=Item#1: its before is unknown",
    "'Big' at i=Item#2: its before is unknown"])); // no size: whether it exceeds 2 is unknown
  const everything = new T.Session(shelf("bolt"), [new T.Transform("All", over(true), over(i.get("size").gt(2n)), { rewrite: size })]);
  assert(everything.candidates().length === 0 && same(everything.undecided, ["'All' at i=Item#0: its after is unknown"])); // before: anything
  const first = session.candidates()[0] as T.Candidate;
  assert(first.open.length === 0 && first.score === 0 && (first.match["i"] as any).name === "bolt");
  store.Item(first.match["i"]).label("BOLT").update(); // outside the session: found again only after a step
  assert(equal(seen(free), [["Free", "bolt", { on: false }], ["Free", "bolt", { on: true }]]) && same((free[0] as T.Candidate).open, ["n"])); // not options
  const answered = (free[0] as T.Candidate).answer({ n: 3n });
  assert(answered.open.length === 0 && equal(answered.arguments, { on: false, n: 3n }) && same((free[0] as T.Candidate).open, ["n"]));
  raises(TypeError, () => (free[0] as T.Candidate).answer({ on: true }), "'Free' has no open parameter 'on'");
}

## TRF-03 · Taking a step: one decision, resolved linearly; refusals; a rewrite must establish its after

In [ ]:
{
  const store = shelf("bolt", "nut");
  const session = new T.Session(store, [Label, Size]);
  const boltUpper = session.candidates()[0] as T.Candidate;
  session.take(boltUpper);
  assert(store.extent("Item")[0].name === "nut"); // the rewrite moved bolt last in its extent; labels keep their first order
  assert(equal(seen(session.candidates()), [["Label", "nut", { case: "upper" }], ["Label", "nut", { case: "lower" }],
    ["Size", "bolt", {}]])); // bolt's other case is no longer enabled
  raises(ValueError, () => session.take(boltUpper), "'Label' is not enabled for that match");
  const sizeBolt = session.candidates()[2] as T.Candidate;
  raises(ValueError, () => session.take(sizeBolt), "'Size' has open parameters ['size']");
  session.take(sizeBolt.answer({ size: 3n }));
  assert((boltUpper.match["i"] as any).label === "BOLT" && (boltUpper.match["i"] as any).size === 3n);
  raises(TypeError, () => session.step_in(session.candidates()[0] as T.Candidate), "'Label' is not composite: take it");
  raises(TypeError, () => new T.Session(shelf("bolt"), [Finish]).take(new T.Session(shelf("bolt"), [Finish]).candidates()[0] as T.Candidate),
    "'Finish' is composite: step in or over it");
  const broken = new T.Transform("Broken", over(i.has("name")), over(i.has("label")), { rewrite: () => null });
  const lazy = new T.Session(shelf("bolt"), [broken]);
  raises(ValueError, () => lazy.take(lazy.candidates()[0] as T.Candidate), "'Broken' did not establish its after at i=Item#0");
}

## TRF-04 · A policy ranks candidates: a recommendation, or the decision where the caller steps over

In [ ]:
{
  const store = shelf("bolt", "nut");
  const session = new T.Session(store, [Label, Tag, Size]);
  const policy = new T.Policy(new T.Clause("Label", { case: "lower" }, 2), new T.Clause("Tag"), new T.Clause("Size", { size: 5n }));
  const ranked = session.candidates(policy);
  assert(equal(ranked.map((c) => [c.transform.name, (c.match["i"] as any).name, c.arguments, c.score]), [
    ["Label", "bolt", { case: "lower" }, 2], ["Label", "nut", { case: "lower" }, 2],
    ["Tag", "bolt", {}, 1], ["Tag", "nut", {}, 1],
    ["Label", "bolt", { case: "upper" }, 0], ["Label", "nut", { case: "upper" }, 0]])); // by score, then in order
  assert(session.step_over(policy) === true && store.extent("Item")[1].label === "bolt"); // the policy's first
  assert(session.run(policy) === 3 && session.done); // nut's label, then each size, which the clause answers
  assert(equal(store.extent("Item").map((item: any) => [item.label, item.size]), [["bolt", 5n], ["nut", 5n]]));
  assert(same(session.steps.map((step) => step.by), ["policy", "policy", "policy", "policy"]));
  const cautious = new T.Session(shelf("bolt"), [Label, Size]);
  assert(cautious.step_over(new T.Policy(new T.Clause("Size"))) === false); // no clause weighs Label: not the policy's to take
  cautious.take(cautious.candidates()[0] as T.Candidate);
  assert(cautious.step_over(new T.Policy(new T.Clause("Size"))) === false && !cautious.done); // it cannot answer size
  assert(cautious.run(new T.Policy(new T.Clause("Size", { size: 1n }))) === 1 && cautious.done && cautious.step_over(policy) === false);
  assert(equal(new T.Policy(new T.Clause("Free", { on: true, n: 7n })).answers(free[1] as T.Candidate), { n: 7n })); // only open parameters are answered
}

## TRF-05 · Composite transforms: stepped into, the caller decides inside; stepped over, the policy does

In [ ]:
{
  const store = shelf("bolt", "nut");
  const session = new T.Session(store, [Finish]);
  assert(equal(seen(session.candidates()), [["Finish", "bolt", {}], ["Finish", "nut", {}]]));
  const inner = session.step_in(session.candidates()[0] as T.Candidate);
  assert(equal(seen(inner.candidates()), [["Label", "bolt", { case: "upper" }], ["Label", "bolt", { case: "lower" }]])); // its match only
  raises(ValueError, () => session.candidates(), "'Finish' is open: take its steps first");
  inner.take(inner.candidates()[1] as T.Candidate);
  inner.take((inner.candidates()[0] as T.Candidate).answer({ size: 2n }));
  assert(inner.done && equal(seen(session.candidates()), [["Finish", "nut", {}]])); // recorded once its session is done
  assert(session.step_over(new T.Policy(new T.Clause("Finish"), new T.Clause("Label", { case: "upper" }), new T.Clause("Size", { size: 4n }))));
  assert(session.done && equal(session.steps.map((s) => [s.transform, s.by, s.mode, s.steps.map((x) => [x.transform, x.by])]), [
    ["Finish", "caller", "in", [["Label", "caller"], ["Size", "caller"]]],
    ["Finish", "policy", "over", [["Label", "policy"], ["Size", "policy"]]]]));
  const Wrap = new T.Transform("Wrap", over(i.has("name")), over(i.has("label")), { parameters: [(p) => p.name("tag")], parts: [Label] });
  const wrapped = new T.Session(shelf("bolt"), [Wrap]);
  raises(ValueError, () => wrapped.step_in(wrapped.candidates()[0] as T.Candidate), "'Wrap' has open parameters ['tag']");
  const partial = new T.Session(shelf("bolt"), [Finish]);
  assert(partial.step_over(new T.Policy(new T.Clause("Finish"), new T.Clause("Label", { case: "lower" }))) === true); // opened, then stuck
  assert(!partial.done && partial.step_over(new T.Policy(new T.Clause("Size", { size: 1n }))) === true); // steps inside
  assert(partial.done && partial.steps[0]?.mode === "over" && same((partial.steps[0] as T.Step).steps.map((x) => x.transform), ["Label", "Size"]));
}

## TRF-06 · The trace is data: written byte-identically, read back, validated; two runs write the same

In [ ]:
function finished(names: string[]): T.Session {
  const store = shelf(...names);
  const session = new T.Session(store, [Finish, Tag]);
  session.step_in(session.candidates()[0] as T.Candidate);
  session.run(new T.Policy(new T.Clause("Finish"), new T.Clause("Label", { case: "upper" }), new T.Clause("Size", { size: 2n })));
  return session;
}

{
  const traces = T.register(new Proxies.OfStore());
  assert(T.register(traces) === traces); // once is enough
  const session = finished(["bolt", "nut"]);
  const text = SchemaJSON.ToJSON(traces)(T.Trace, session.trace(traces) as never);
  assert(text === SchemaJSON.ToJSON(traces)(T.Trace, finished(["bolt", "nut"]).trace(traces) as never)); // deterministic
  assert(text === (
    '{"root": "s0", "objects": {"s0": {"steps": ['
    + '{"transform": "Finish", "match": [{"symbol": "i", "element": "Item#0"}], "by": "caller", "mode": "in", "steps": ['
    + '{"transform": "Label", "match": [{"symbol": "i", "element": "Item#0"}], "arguments": [{"name": "case", "value": {"str": "upper"}}], "by": "policy"}, '
    + '{"transform": "Size", "match": [{"symbol": "i", "element": "Item#0"}], "arguments": [{"name": "size", "value": {"int": 2}}], "by": "policy"}]}, '
    + '{"transform": "Finish", "match": [{"symbol": "i", "element": "Item#1"}], "by": "policy", "mode": "over", "steps": ['
    + '{"transform": "Label", "match": [{"symbol": "i", "element": "Item#1"}], "arguments": [{"name": "case", "value": {"str": "upper"}}], "by": "policy"}, '
    + '{"transform": "Size", "match": [{"symbol": "i", "element": "Item#1"}], "arguments": [{"name": "size", "value": {"int": 2}}], "by": "policy"}]}]}}}'));
  const back = SchemaJSON.FromJSON(traces)(T.Trace, text);
  assert(SchemaJSON.ToJSON(traces)(T.Trace, back as never) === text && SV.Validate(traces)(T.Trace, back).length === 0);
  const Note = new T.Transform("Note", over(i.has("name")), over(i.has("label")), { parameters: [(p) => p.name("note")],
    rewrite: (store, match) => store.Item(match["i"]).label("noted").update() });
  const noted = new T.Session(shelf("bolt"), [Note]);
  noted.take((noted.candidates()[0] as T.Candidate).answer({ note: null }));
  raises(TypeError, () => noted.trace(traces), "argument 'note' is not a native value: None");
}

## TRF-07 · Transforms match schemas, the objects of a store of schemas (mbse-schemas' `Reflection`)

In [ ]:
{
  const Phone = new S.OfObject.Builder().name("Phone").properties(native("number")).create();
  const Contact = new S.OfObject.Builder().name("Contact").ref().properties(native("name"), (p) => p.name("home").of(Phone)).create();
  const s = E.variable("s");
  const named = (constraint: unknown) => new P.OfPredicate.Builder().symbols({ s: R.META.get(S.OfObject.Data) as S.OfObject.Data })
    .requires(constraint).create();
  const Document = new T.Transform("Document", named(s.has("name")), named(s.has("description")),
    { rewrite: (store, match) => { (match["s"] as any).description = `A ${(match["s"] as any).name}`; } });
  const schemas = R.store([Contact, Phone]);
  const documenting = new T.Session(schemas, [Document]);
  assert(same(documenting.candidates().map((c) => c.match["s"]), [Contact, Phone])); // the schemas themselves
  assert(documenting.run(new T.Policy(new T.Clause("Document"))) === 2 && documenting.done);
  assert(Contact.description === "A Contact" && Phone.description === "A Phone");
  assert(equal(documenting.steps.map((step) => step.match), [[["s", "Schemas.Object#0"]], [["s", "Schemas.Object#1"]]]));
}